# Bölüm 1 – Machine Learning Manzarası (The Machine Learning Landscape)

> **Türkçe Açıklamalı Notebook**  
> Orijinal kaynak: *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (Aurélien Géron)  
> Bu notebook, bölüm 1'deki tüm kod örneklerini ve alıştırma çözümlerini içermektedir.  
> Ayrıca `lifesat.csv` veri setinin nasıl oluşturulduğunu ve bölümdeki görsellerin nasıl üretildiğini göstermektedir.

---

## Bu Bölümde Ne Öğreneceğiz?

Bu bölüm, Machine Learning'e (Makine Öğrenmesi) geniş bir bakış açısı sunar:

| Konu | Açıklama |
|------|----------|
| **Machine Learning Nedir?** | Veriden öğrenen sistemler inşa etme sanatı |
| **Supervised vs Unsupervised** | Gözetimli ve gözetimsiz öğrenme farkları |
| **Linear Regression** | Basit doğrusal model ile tahmin |
| **K-Nearest Neighbors** | Komşuluğa dayalı model |
| **Overfitting & Underfitting** | Aşırı ve yetersiz uyum sorunları |
| **Regularization** | Modeli kısıtlayarak genelleme iyileştirme |
| **GDP vs Life Satisfaction** | Gerçek dünya verisiyle end-to-end örnek |

---

## Temel Machine Learning Kavramları

### Machine Learning Nedir?
Arthur Samuel'in (1959) tanımı:
> "ML, bilgisayarlara açıkça programlanmadan öğrenme yeteneği kazandıran çalışma alanıdır."

Tom Mitchell'in (1997) daha resmi tanımı:
> "Bir bilgisayar programının deneyim E ile görev T'de P performans ölçütüne göre iyileşmesi durumunda, T görevi için E deneyiminden P ile öğrendiği söylenir."

### ML Sistemlerinin Türleri

**Öğrenme Gözetimi Açısından:**
- **Supervised Learning** (Gözetimli): Etiketli veriyle eğitim — sınıflandırma, regresyon
- **Unsupervised Learning** (Gözetimsiz): Etiketsiz veriyle desen bulma — kümeleme, boyut indirgeme
- **Semi-supervised Learning** (Yarı gözetimli): Az etiketli + çok etiketsiz veri
- **Self-supervised Learning** (Öz gözetimli): Etiketler veriden türetilir
- **Reinforcement Learning** (Pekiştirmeli): Ajan, ödül/ceza ile öğrenir

**Öğrenme Modu Açısından:**
- **Batch Learning**: Tüm veriyle bir kez eğitim, sonra statik
- **Online Learning**: Veri akışıyla sürekli güncelleme

**Genelleme Yöntemi Açısından:**
- **Instance-based**: Yeni örnekleri eğitim örnekleriyle karşılaştır (k-NN)
- **Model-based**: Eğitim verisinden model parametreleri öğren (Linear Regression)


# 1. Kurulum (Setup)

## 1.1 Python Sürüm Kontrolü

Bu proje **Python 3.10 veya üzerini** gerektirir. `sys.version_info` ile mevcut Python sürümü kontrol edilir; koşul sağlanmazsa `AssertionError` fırlatılır.

In [ ]:
import sys

# Python sürümünün 3.10 veya üzeri olduğunu doğrula
# sys.version_info: (major, minor, micro, ...) şeklinde tuple döndürür
# (3, 10) ile karşılaştırma: major=3, minor≥10 olması yeterli
assert sys.version_info >= (3, 10)

## 1.2 Scikit-Learn Sürüm Kontrolü

Bu notebook **Scikit-Learn 1.6.1 veya üzerini** gerektirir.  
Scikit-Learn, Python'ın en kapsamlı Machine Learning kütüphanesidir: sınıflandırma, regresyon, kümeleme, boyut indirgeme, model seçimi ve ön işleme araçları içerir.

`packaging.version.Version` ile sürüm karşılaştırması güvenli biçimde yapılır (string karşılaştırması hatalı sonuç verebilir: "1.10" < "1.9" gibi).


In [ ]:
from packaging.version import Version  # Güvenli sürüm karşılaştırması için
import sklearn                           # Scikit-Learn ana modülü

# Scikit-Learn sürümünün 1.6.1 veya üzeri olduğunu doğrula
assert Version(sklearn.__version__) >= Version("1.6.1")

## 1.3 Matplotlib Grafik Ayarları

Grafiklerin daha okunabilir ve tutarlı görünmesi için varsayılan yazı tipi boyutlarını ayarlıyoruz. `plt.rc()` ile global stili bir kez tanımlamak yeterlidir; sonraki tüm grafikler bu ayarları kullanır.

In [ ]:
import matplotlib.pyplot as plt

# Tüm grafiklerde kullanılacak varsayılan yazı boyutlarını ayarla
plt.rc('font', size=12)             # Genel yazı boyutu (başlık, etiket dışı)
plt.rc('axes', labelsize=14, titlesize=14)  # Eksen etiketleri ve grafik başlığı
plt.rc('legend', fontsize=12)       # Lejand (açıklama kutusu) yazı boyutu
plt.rc('xtick', labelsize=10)       # X ekseni ölçek etiketi boyutu
plt.rc('ytick', labelsize=10)       # Y ekseni ölçek etiketi boyutu

# 2. Kod Örneği 1-1: GDP ve Yaşam Memnuniyeti

## Problemin Tanımı

Bu örnekte şu soruyu yanıtlamaya çalışıyoruz:
> "Bir ülkenin kişi başı GDP'si ile o ülke vatandaşlarının yaşam memnuniyeti arasında nasıl bir ilişki var?"

## Veri Kaynakları

- **GDP per capita**: Dünya Bankası verisi (Our World in Data üzerinden)
- **Life satisfaction**: OECD Better Life Index (BLI) anketi — 0-10 ölçeği

## Yaklaşım: Model-Based Learning

Veriye bir **Linear Regression** (Doğrusal Regresyon) modeli uyduruyoruz:

```
yaşam_memnuniyeti = θ₀ + θ₁ × gdp_per_capita
```

- **θ₀** (intercept/bias): Doğrunun y-eksenini kestiği nokta
- **θ₁** (slope/eğim): GDP'nin 1 birim artmasına karşılık memnuniyetteki değişim

Model eğitildikten sonra yeni bir ülkenin (Puerto Rico) GDP'sini girerek yaşam memnuniyetini **tahmin edebiliriz**.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# ─── VERİYİ İNDİR VE HAZIRLA ────────────────────────────────────────────────
data_root = "https://github.com/ageron/data/raw/main/"

# lifesat.csv: Her satır bir ülke, sütunlar GDP per capita ve Life satisfaction
# Bu dosya önceden hazırlanmış temiz bir veri setidir
lifesat = pd.read_csv(data_root + "lifesat/lifesat.csv")

# Feature matrix X: Girdi değişkeni (bağımsız değişken)
# [["GDP per capita (USD)"]]: Çift köşeli parantez → DataFrame (2D array) döndürür
# .values: NumPy array'e çevir (Scikit-Learn bu formatı bekler)
X = lifesat[["GDP per capita (USD)"]].values  # shape: (n_ülke, 1)

# Target vector y: Çıktı değişkeni (bağımlı değişken, tahmin etmek istediğimiz)
y = lifesat[["Life satisfaction"]].values     # shape: (n_ülke, 1)

# ─── VERİYİ GÖRSELLEŞTIR ────────────────────────────────────────────────────
# DataFrame.plot(kind='scatter'): Pandas'ın scatter plot kısayolu
# grid=True: Arka plan ızgara çizgileri ekle
lifesat.plot(kind='scatter', grid=True,
             x="GDP per capita (USD)", y="Life satisfaction")

# Eksen sınırlarını belirle: [x_min, x_max, y_min, y_max]
# Odak bölgesi: GDP $23,500-$62,500 arası, memnuniyet 4-9 arası
plt.axis([23_500, 62_500, 4, 9])
plt.show()

# ─── MODEL SEÇ: Linear Regression ───────────────────────────────────────────
# Linear Regression: y = θ₀ + θ₁*x formülüne en iyi θ değerlerini bul
# "En iyi" → Eğitim verisindeki Mean Squared Error (MSE) en aza indir
model = LinearRegression()

# ─── MODELİ EĞİT ─────────────────────────────────────────────────────────────
# fit(X, y): X ve y'deki ilişkiden θ₀ ve θ₁ parametrelerini öğren
# Algoritma: Normal Equations veya Gradient Descent (büyük veri için)
model.fit(X, y)

# ─── TAHMİN YAP ───────────────────────────────────────────────────────────────
# Puerto Rico'nun 2020 kişi başı GDP'si: $33,442.8
# Model hiç Puerto Rico verisi görmeden tahmin yapıyor!
X_new = [[33_442.8]]
print(model.predict(X_new))  # [[6.01610329]] — 0-10 ölçeğinde yaşam memnuniyeti

## 2.1 K-Nearest Neighbors ile Aynı Problem

Scikit-Learn'in en güçlü özelliklerinden biri **tutarlı API** tasarımıdır:  
Tüm modeller `fit()`, `predict()`, `transform()` gibi aynı metodları kullanır.

Bu sayede bir modeli başka bir modelle değiştirmek yalnızca **2 satır kod** değiştirmek demektir!

### K-Nearest Neighbors (k-NN) Nedir?

**Instance-based** öğrenme algoritması — eğitim verisini ezberleme:
1. Yeni bir örnek geldiğinde, eğitim setindeki en yakın `k` komşuyu bul
2. Bu `k` komşunun ortalama hedef değerini tahmin olarak döndür

k=3: 3 en yakın komşunun GDP değeri, Puerto Rico'ya en yakın olan 3 ülkenin yaşam memnuniyetinin ortalamasını verir.

```
Linear Regression → Model tabanlı (parametrik)
K-NN              → Instance tabanlı (non-parametrik)
```


In [ ]:
# K-Nearest Neighbors Regressor seç
# n_neighbors=3: En yakın 3 komşuyu kullan
from sklearn.neighbors import KNeighborsRegressor

# Orijinal LinearRegression yerine sadece bu 2 satırı değiştirdik!
# X ve y önceki hücreden mevcut: Linear Regression ile aynı veri
model = KNeighborsRegressor(n_neighbors=3)

# MODELİ EĞİT — API tamamen aynı: fit(X, y)
# k-NN "eğitimi" aslında sadece veriyi bellekte tutar (lazy learning)
model.fit(X, y)

# TAHMİN YAP — API tamamen aynı: predict(X_new)
# k-NN: Puerto Rico'ya GDP açısından en yakın 3 ülkeyi bul, memnuniyetlerinin ortalamasını al
print(model.predict(X_new))  # [[5.73333333]] — 3 komşunun ortalaması

# 3. Veri Seti Oluşturma ve Görselleştirme

> **Not**: Bu bölüm, `lifesat.csv` veri setinin nasıl oluşturulduğunu ve kitaptaki görsellerin nasıl üretildiğini gösterir. Temel Machine Learning kavramlarını öğrenmek istiyorsanız bu bölümü atlayabilirsiniz.

## Ham Veriler

1. **oecd_bli.csv**: OECD Better Life Index verisi
   - Ülkelere göre yaşam kalitesi metrikleri (eğitim, sağlık, mutluluk vb.)
   - `INEQUALITY` sütunu: "TOT" = tüm nüfus ortalaması
   
2. **gdp_per_capita.csv**: Dünya Bankası GDP verisi
   - Ülkelere ve yıllara göre kişi başı GDP değerleri (USD)
   - 2020 yılı verisi kullanılacak

## İşlem Adımları

1. Ham CSV dosyalarını indir
2. GDP verisini 2020 yılına filtrele ve temizle
3. BLI verisini pivot table'a dönüştür ve "Life satisfaction" sütununu al
4. İki veri setini ülke adına göre birleştir (`merge`)
5. Overfitting demonstrasyonu için GDP aralığını sınırlandır
6. Temiz veriyi `lifesat.csv` olarak kaydet


## 3.1 Ham Veri Dosyalarını İndirme

In [ ]:
from pathlib import Path    # Platform bağımsız dosya yolu işlemleri
import urllib.request         # URL'den dosya indirme

# Veri dizini oluştur: datasets/lifesat/
# parents=True: Üst dizinler yoksa da oluştur (mkdir -p gibi)
# exist_ok=True: Dizin zaten varsa hata verme
datapath = Path() / "datasets" / "lifesat"
datapath.mkdir(parents=True, exist_ok=True)

data_root = "https://github.com/ageron/data/raw/main/"

# Her iki CSV dosyasını indir (sadece yoksa — önbellek mantığı)
for filename in ("oecd_bli.csv", "gdp_per_capita.csv"):
    if not (datapath / filename).is_file():  # Dosya yoksa indir
        print("Downloading", filename)
        url = data_root + "lifesat/" + filename
        urllib.request.urlretrieve(url, datapath / filename)  # URL → yerel dosya

## 3.2 CSV Dosyalarını Yükleme

In [ ]:
# Ham CSV dosyalarını Pandas DataFrame'e yükle
oecd_bli        = pd.read_csv(datapath / "oecd_bli.csv")        # OECD BLI verisi
gdp_per_capita  = pd.read_csv(datapath / "gdp_per_capita.csv")  # GDP verisi

## 3.3 GDP Verisini Ön İşleme

GDP verisi yıllara göre uzun formatta (long format) gelir. 2020 yılını filtreleyip kullanışlı formata getiriyoruz.


In [ ]:
# Hedef yıl ve sütun adları
gdp_year    = 2020
gdppc_col   = "GDP per capita (USD)"
lifesat_col = "Life satisfaction"

# ─── 2020 YILINI FİLTRELE ─────────────────────────────────────────────────────
# GDP verisi: Her satır bir ülke-yıl çifti
# Yalnızca 2020 satırlarını tut (boolean indexing)
gdp_per_capita = gdp_per_capita[gdp_per_capita["Year"] == gdp_year]

# Artık ihtiyaç duyulmayan sütunları sil
# "Code": Ülke kodu (ISO), "Year": Artık gereksiz (hepsi 2020)
# axis=1: Sütun silme (axis=0 satır silerdi)
gdp_per_capita = gdp_per_capita.drop(["Code", "Year"], axis=1)

# Sütun adlarını yeniden adlandır
gdp_per_capita.columns = ["Country", gdppc_col]

# "Country" sütununu index yap (merge işlemini kolaylaştırır)
# inplace=True: Yeni bir DataFrame oluşturmak yerine mevcut olanı değiştir
gdp_per_capita.set_index("Country", inplace=True)

# İlk 5 satırı göster
gdp_per_capita.head()

## 3.4 OECD BLI Verisini Ön İşleme

BLI verisi "uzun format"ta gelir: Her satır bir ülke-gösterge-yıl üçlüsü.  
Bunu "geniş formata" (wide format) dönüştürüp yalnızca "Life satisfaction" göstergesini alıyoruz.


In [ ]:
# ─── EŞİTSİZLİK FİLTRESİ ──────────────────────────────────────────────────────
# INEQUALITY sütunu: "TOT" = tüm nüfus ortalaması
#                    "HIGH"/"LOW" = gelir gruplarına göre ayrışım
# Sadece toplam (TOT) satırlarını tut — ülke geneli ortalama değerler
oecd_bli = oecd_bli[oecd_bli["INEQUALITY"] == "TOT"]

# ─── PIVOT TABLE ──────────────────────────────────────────────────────────────
# Uzun format → Geniş format dönüşümü
# index="Country": Her satır bir ülke
# columns="Indicator": Her sütun bir gösterge (Life satisfaction, Education, vb.)
# values="Value": Hücre değerleri
# Sonuç: Her ülke için tüm BLI göstergelerini içeren tablo
oecd_bli = oecd_bli.pivot(index="Country", columns="Indicator", values="Value")

# İlk 5 satırı göster (çok sütun var)
oecd_bli.head()

## 3.5 İki Veri Setini Birleştirme

GDP ve yaşam memnuniyeti verilerini `pd.merge()` ile ülke adına göre birleştiriyoruz.


In [ ]:
# ─── İKİ VERİ SETİNİ BİRLEŞTİR ───────────────────────────────────────────────
# pd.merge: SQL JOIN benzeri birleştirme
# left=oecd_bli: Sol tablo (OECD BLI)
# right=gdp_per_capita: Sağ tablo (GDP)
# left_index=True, right_index=True: Her iki tablonun index'i (Country) üzerinden birleştir
# Varsayılan: inner join — her iki tabloda da bulunan ülkeler
full_country_stats = pd.merge(left=oecd_bli, right=gdp_per_capita,
                              left_index=True, right_index=True)

# GDP'ye göre artan sıraya diz (görsel için kullanışlı)
# inplace=True: Orijinal DataFrame'i değiştir, yeni DataFrame oluşturma
full_country_stats.sort_values(by=gdppc_col, inplace=True)

# Sadece ilgilendiğimiz 2 sütunu seç: GDP per capita ve Life satisfaction
full_country_stats = full_country_stats[[gdppc_col, lifesat_col]]

# İlk 5 satırı göster
full_country_stats.head()

## 3.6 Overfitting Demonstrasyonu için Veri Kısıtlama

Kitapta, overfitting riskini göstermek için yalnızca belirli bir GDP aralığındaki ülkeler kullanılır.  
Bu "eksik" ülkeler sonraki figürlerde gösterilecek ve doğrusal trendi bozdukları görülecek.


In [ ]:
# Gösterim için GDP aralığı: $23,500 - $62,500 arası ülkeler
# Bu aralık dışındaki ülkeler (çok fakir veya çok zengin) kasıtlı olarak çıkarıldı
# Amaç: Bu eksik verilerle yanlış sonuçlara ulaşma riskini göstermek
min_gdp = 23_500  # Alt sınır (USD)
max_gdp = 62_500  # Üst sınır (USD)

# Koşullu filtreleme: Her iki koşul da sağlanmalı (&: ve)
# Daha önce sort_values yapıldığından ülkeler GDP sırasına göre dizili
country_stats = full_country_stats[(full_country_stats[gdppc_col] >= min_gdp) &
                                   (full_country_stats[gdppc_col] <= max_gdp)]
country_stats.head()

## 3.7 Veriyi CSV Olarak Kaydet

In [ ]:
# Temizlenmiş ve kısıtlanmış veriyi kaydet
# lifesat.csv: Ana kod örneğinde kullanılan veri seti (sadece belirli GDP aralığı)
country_stats.to_csv(datapath / "lifesat.csv")

# lifesat_full.csv: Tüm ülkeleri içeren tam veri seti (overfitting gösterimi için)
full_country_stats.to_csv(datapath / "lifesat_full.csv")

print("Dosyalar kaydedildi:")
print(f"  - {datapath / 'lifesat.csv'} ({len(country_stats)} ülke)")
print(f"  - {datapath / 'lifesat_full.csv'} ({len(full_country_stats)} ülke)")

## 3.8 Ülke Etiketleri ile Scatter Plot

Veriyi görselleştirirken seçili ülkelerin adlarını okla işaretliyoruz.  
Bu, doğrusal trendin gerçekten anlamlı olup olmadığını gözle değerlendirmemizi sağlar.


In [ ]:
# GDP vs Life Satisfaction scatter plot (ülke etiketleri ile)
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

# Grafik sınırları
min_life_sat = 4
max_life_sat = 9

# Etiketlenecek ülkeler ve metin konumları (x, y): Okun işaret edeceği yer değil
# Metin kutusu konumu — ok çizgisi noktayı gerçek konumuna bağlar
position_text = {
    "Turkey":        (29_500, 4.2),
    "Hungary":       (28_000, 6.9),
    "France":        (40_000, 5),
    "New Zealand":   (28_000, 8.2),
    "Australia":     (50_000, 5.5),
    "United States": (59_000, 5.3),
    "Denmark":       (46_000, 8.5)
}

for country, pos_text in position_text.items():
    # Gerçek veri noktasının koordinatlarını al
    pos_data_x = country_stats[gdppc_col].loc[country]    # GDP değeri
    pos_data_y = country_stats[lifesat_col].loc[country]  # Memnuniyet değeri

    # "United States" → "US" olarak kısalt (grafik alanında yer tasarrufu)
    country = "US" if country == "United States" else country

    # plt.annotate: Ok + metin açıklaması ekle
    # xy: Okun gösterdiği nokta (gerçek veri koordinatları)
    # xytext: Metin kutusunun konumu
    # arrowprops: Ok stil parametreleri
    plt.annotate(country,
                 xy=(pos_data_x, pos_data_y),   # Okun ucu
                 xytext=pos_text,                # Metin konumu
                 fontsize=12,
                 arrowprops=dict(
                     facecolor='black',   # Ok rengi
                     width=0.5,           # Ok gövde genişliği
                     shrink=0.08,         # Ok ucunun noktadan uzaklığı
                     headwidth=5          # Ok başı genişliği
                 ))
    plt.plot(pos_data_x, pos_data_y, "ro")  # Kırmızı nokta ile işaretle

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

## 3.9 Öne Çıkan Ülkelerin Veri Tablosu

In [ ]:
# Etiketlenen ülkelerin GDP ve yaşam memnuniyeti değerlerini tablo olarak göster
# GDP'ye göre sırala: En düşükten en yükseğe
highlighted_countries = country_stats.loc[list(position_text.keys())]
highlighted_countries[[gdppc_col, lifesat_col]].sort_values(by=gdppc_col)
# Bu tablo: Düşük GDP → düşük memnuniyet, Yüksek GDP → yüksek memnuniyet örüntüsü

## 3.10 Farklı Linear Model Parametre Denemeleri

Bu görsel, model parametrelerinin (θ₀ ve θ₁) veriyi nasıl fit ettiğini gösterir.  
Üç farklı parametre kombinasyonu deniyoruz — hangisinin "en iyi" olduğunu göreceğiz.

```
Kırmızı: θ₀=4.2,  θ₁=0      → Düz yatay çizgi (slope yok)
Yeşil:   θ₀=10,   θ₁=-9e-5  → Yanlış yön (negatif slope)
Mavi:    θ₀=3,    θ₁=8e-5   → Mantıklı pozitif slope
```

Doğrusal model denklemi: `y = θ₀ + θ₁ × x`


In [ ]:
# Scatter plot üzerine 3 farklı doğrusal model çiz
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

# x ekseni için 1000 eşit aralıklı nokta (düzgün çizgi için)
X = np.linspace(min_gdp, max_gdp, 1000)

# ─── KIRMIZI: Düz çizgi — slope yok (kötü model) ─────────────────────────────
w1, w2 = 4.2, 0
plt.plot(X, w1 + w2 * 1e-5 * X, "r")          # "r": kırmızı çizgi
plt.text(40_000, 4.9, fr"$	heta_0 = {w1}$", color="r")
plt.text(40_000, 4.4, fr"$	heta_1 = {w2}$", color="r")

# ─── YEŞİL: Negatif slope — yanlış yön (kötü model) ─────────────────────────
w1, w2 = 10, -9
plt.plot(X, w1 + w2 * 1e-5 * X, "g")          # "g": yeşil çizgi
plt.text(26_000, 8.5, fr"$	heta_0 = {w1}$", color="g")
plt.text(26_000, 8.0, fr"$	heta_1 = {w2} 	imes 10^{{-5}}$", color="g")

# ─── MAVİ: Pozitif slope — mantıklı yön ──────────────────────────────────────
w1, w2 = 3, 8
plt.plot(X, w1 + w2 * 1e-5 * X, "b")          # "b": mavi çizgi
plt.text(48_000, 8.5, fr"$	heta_0 = {w1}$", color="b")
plt.text(48_000, 8.0, fr"$	heta_1 = {w2} 	imes 10^{{-5}}$", color="b")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()
# Sonuç: Mavi çizgi en iyi görünüyor ama optimal θ değerleri hâlâ elle belirlendi
# Bir sonraki adımda Scikit-Learn bu değerleri otomatik bulacak

## 3.11 Linear Regression ile Optimal Parametreleri Bulma

Elle deneme yerine Scikit-Learn'in `LinearRegression` modeli, **Normal Equations** (Kapalı Form Çözümü) veya **Gradient Descent** ile optimal θ parametrelerini otomatik olarak bulur.

**Cost Function (Maliyet Fonksiyonu)**: MSE (Mean Squared Error)
```
MSE(θ) = (1/n) × Σ (y_pred_i - y_true_i)²
```

Minimum MSE veren θ₀ ve θ₁ değerleri:
- `lin1.intercept_`: θ₀ (y-eksen kesişimi)
- `lin1.coef_`: θ₁ (eğim / slope)


In [ ]:
from sklearn import linear_model

# ─── EĞİTİM VERİSİNİ HAZIRLA ─────────────────────────────────────────────────
# .values: DataFrame → NumPy array (Scikit-Learn tercih eder)
X_sample = country_stats[[gdppc_col]].values   # shape: (n, 1) — 2D gerekli
y_sample = country_stats[[lifesat_col]].values # shape: (n, 1)

# ─── MODELİ OLUŞTUR VE EĞİT ──────────────────────────────────────────────────
lin1 = linear_model.LinearRegression()
lin1.fit(X_sample, y_sample)  # Normal Equations ile optimal θ bul

# Öğrenilen parametreleri çıkar
# intercept_: [0] ile skaler al (çünkü çoklu çıkış olmadığında 1D array)
# coef_: [0][0] ile skaler al (2D array: [n_outputs, n_features])
t0, t1 = lin1.intercept_[0], lin1.coef_[0][0]

# Öğrenilen parametreleri yazdır
print(f"θ0={t0:.2f}, θ1={t1:.2e}")
# θ₁ çok küçük bir sayı (≈ 5e-05): GDP 1 dolar artınca memnuniyet 0.00005 artıyor
# Ama GDP 10,000 dolar artınca memnuniyet 0.5 artıyor (makul bir ilişki!)

In [ ]:
# Optimal doğrusal modeli scatter plot üzerine çiz
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

X = np.linspace(min_gdp, max_gdp, 1000)

# Öğrenilen parametrelerle doğruyu çiz: y = t0 + t1*X
plt.plot(X, t0 + t1 * X, "b")

# Parametre değerlerini grafik üzerine yaz
plt.text(max_gdp - 20_000, min_life_sat + 1.9,
         fr"$	heta_0 = {t0:.2f}$", color="b")
plt.text(max_gdp - 20_000, min_life_sat + 1.3,
         fr"$	heta_1 = {t1 * 1e5:.2f} 	imes 10^{{-5}}$", color="b")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()
# Bu çizgi, veriyi en iyi temsil eden doğrusal fonksiyon

## 3.12 Puerto Rico için Tahmin Yapma

Modelimiz eğitildikten sonra, eğitim setinde **hiç görmediği** yeni bir ülke için tahmin yapabiliriz.


In [ ]:
# Puerto Rico'nun 2020 kişi başı GDP değerini al
# gdp_per_capita.loc["Puerto Rico"]: Index'e göre satır seçimi
puerto_rico_gdp_per_capita = gdp_per_capita[gdppc_col].loc["Puerto Rico"]
puerto_rico_gdp_per_capita  # ~$33,442 USD

In [ ]:
# Puerto Rico için yaşam memnuniyeti tahmini
# lin1.predict([[değer]]): 2D array gerekli (çift köşeli parantez)
# [0, 0]: 2D sonuç matrisinden tek skaler değer çıkar
puerto_rico_predicted_life_satisfaction = lin1.predict(
    [[puerto_rico_gdp_per_capita]])[0, 0]

puerto_rico_predicted_life_satisfaction  # ~6.01 (0-10 ölçeği)
# Gerçek OECD verisi: ~6.4 — model makul ama mükemmel değil

In [ ]:
# Tahmini grafik üzerinde görselleştir
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

X = np.linspace(min_gdp, max_gdp, 1000)
plt.plot(X, t0 + t1 * X, "b")

# Parametre değerleri
plt.text(min_gdp + 22_000, max_life_sat - 1.1,
         fr"$	heta_0 = {t0:.2f}$", color="b")
plt.text(min_gdp + 22_000, max_life_sat - 0.6,
         fr"$	heta_1 = {t1 * 1e5:.2f} 	imes 10^{{-5}}$", color="b")

# Tahmin için dikey kesikli çizgi: x = Puerto Rico GDP
# [x_start, x_end], [y_start, y_end]: Dikey çizgi
plt.plot([puerto_rico_gdp_per_capita, puerto_rico_gdp_per_capita],
         [min_life_sat, puerto_rico_predicted_life_satisfaction], "r--")

# Tahmin değerini metin olarak göster
plt.text(puerto_rico_gdp_per_capita + 1000, 5.0,
         fr"Prediction = {puerto_rico_predicted_life_satisfaction:.2f}",
         color="r")

# Tahmin noktasını kırmızı daire ile işaretle
plt.plot(puerto_rico_gdp_per_capita, puerto_rico_predicted_life_satisfaction, "ro")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

## 3.13 Eksik Veriler ve Temsil Edilmezlik Riski

Modelimiz yalnızca belirli bir GDP aralığındaki ($23,500-$62,500) ülkelerle eğitildi.  
Bu aralık dışındaki ülkeler (çok fakir veya çok zengin) görmezden gelindi.

**Temsil Edilmezlik Problemi (Sampling Bias)**:
Eğer eğitim verisi gerçek dağılımı temsil etmiyorsa, model gerçek dünyada kötü performans gösterir.

Bu bölümde göreceğiz:
- Eksik ülkeler (çok düşük/yüksek GDP) eklendikçe doğrusal trend değişiyor
- Yüksek GDP'li ülkelerde (Lüksemburg, İrlanda vb.) memnuniyet artmıyor
- Bu, doğrusal ilişkinin abartılmış olduğunu gösteriyor


In [ ]:
# Kısıtlanmış aralığın dışındaki ülkeleri bul (eksik veriler)
# Koşul: GDP < 23,500 VEYA (|) GDP > 62,500
missing_data = full_country_stats[(full_country_stats[gdppc_col] < min_gdp) |
                                  (full_country_stats[gdppc_col] > max_gdp)]
missing_data  # Eksik ülkeleri göster (Güney Amerika, Afrika, çok zengin Avrupa)

## 3.14 Eksik Ülkeler için Metin Konumları

In [ ]:
# Eksik ülkeler için grafik metin konumları
# Her ülke: (metin_x_konumu, metin_y_konumu)
position_text_missing_countries = {
    "South Africa": (20_000, 4.2),   # Düşük GDP, düşük memnuniyet
    "Colombia":     (6_000,  8.2),   # Düşük GDP ama yüksek memnuniyet (anomali!)
    "Brazil":       (18_000, 7.8),   # Orta GDP, yüksek memnuniyet (anomali!)
    "Mexico":       (24_000, 7.4),
    "Chile":        (30_000, 7.0),
    "Norway":       (51_000, 6.2),   # Yüksek GDP ama model tahmininden düşük
    "Switzerland":  (62_000, 5.7),
    "Ireland":      (81_000, 5.2),   # Çok yüksek GDP — memnuniyet plato yapıyor
    "Luxembourg":   (92_000, 4.7),   # En yüksek GDP — memnuniyet düşüyor!
}

In [ ]:
# Tam veri seti ile scatter plot (eksik ülkeler dahil)
full_country_stats.plot(kind='scatter', figsize=(8, 3),
                        x=gdppc_col, y=lifesat_col, grid=True)

# Eksik ülkeleri kırmızı kare ile işaretle ve etiketle
for country, pos_text in position_text_missing_countries.items():
    pos_data_x, pos_data_y = missing_data.loc[country]  # Gerçek koordinatlar
    plt.annotate(country,
                 xy=(pos_data_x, pos_data_y),
                 xytext=pos_text, fontsize=12,
                 arrowprops=dict(facecolor='black', width=0.5,
                                 shrink=0.08, headwidth=5))
    plt.plot(pos_data_x, pos_data_y, "rs")  # "rs": kırmızı kare (square)

# ─── KISMİ VERİ MODELİ ───────────────────────────────────────────────────────
# Önceki (kısmi veri) modelin uzatılmış tahmini — kesikli mavi çizgi
X = np.linspace(0, 115_000, 1000)
plt.plot(X, t0 + t1 * X, "b:")  # "b:": mavi noktalı çizgi

# ─── TAM VERİ MODELİ ──────────────────────────────────────────────────────────
# Tüm ülkeler dahil yeni LinearRegression eğit
lin_reg_full = linear_model.LinearRegression()
Xfull = np.c_[full_country_stats[gdppc_col]]   # np.c_: column-wise birleştirme
yfull = np.c_[full_country_stats[lifesat_col]]
lin_reg_full.fit(Xfull, yfull)

# Tam verinin öğrendiği parametreler
t0full, t1full = lin_reg_full.intercept_[0], lin_reg_full.coef_[0][0]

# Tam veri modeli — katı siyah çizgi
X = np.linspace(0, 115_000, 1000)
plt.plot(X, t0full + t1full * X, "k")  # "k": siyah katı çizgi

plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()
# Sonuç: Mavi (kısmi) ve siyah (tam) çizgiler birbirinden önemli ölçüde farklı
# → Temsil edilmezlik, modelin genelleme performansını düşürüyor!

## 3.15 Overfitting Demonstrasyonu — Polynomial Regression

**Overfitting (Aşırı Uyum)**: Model, eğitim verisini çok iyi öğrenir ama yeni veriye kötü geneller.

Daha karmaşık bir model kullanarak overfitting'i görselleştiriyoruz:  
**10. derece polinom** → Her veri noktasına neredeyse mükemmel uyar ama anlamsız dalgalanmalar üretir.

### Pipeline (Ardışık İşlem Hattı)

Scikit-Learn'in `Pipeline` sınıfı birden fazla adımı zincirlememizi sağlar:
1. **PolynomialFeatures**: `x → [x, x², x³, ..., x¹⁰]` (özellik türetme)
2. **StandardScaler**: Özellikleri normalize et (sayısal kararlılık için)
3. **LinearRegression**: Polinom katsayılarını öğren

Bu pipeline, `fit()` ve `predict()` çağrılarında otomatik olarak her adımı sırayla uygular.


In [ ]:
from sklearn import preprocessing
from sklearn import pipeline

# Tam veri scatter plot
full_country_stats.plot(kind='scatter', figsize=(8, 3),
                        x=gdppc_col, y=lifesat_col, grid=True)

# ─── 10. DERECE POLİNOM MODELİ ───────────────────────────────────────────────
# PolynomialFeatures(degree=10): x'ten 10 polinom özellik üret
# [x] → [x, x², x³, ..., x¹⁰]
# include_bias=False: Sabit terim (1) ekleme — LinearRegression zaten ekler
poly = preprocessing.PolynomialFeatures(degree=10, include_bias=False)

# StandardScaler: Her özelliği (μ=0, σ=1) olacak şekilde normalize et
# Polinom özellikleri çok farklı ölçeklerde (x vs x¹⁰) → normalizasyon şart
scaler = preprocessing.StandardScaler()

# Son adım: Polinom özelliklerinden doğrusal regresyon
lin_reg2 = linear_model.LinearRegression()

# Pipeline: 3 adımı sırayla bağla
# fit() → poly.fit_transform → scaler.fit_transform → lin_reg2.fit
# predict() → poly.transform → scaler.transform → lin_reg2.predict
pipeline_reg = pipeline.Pipeline([
    ('poly', poly),       # Adım 1: Polinom özellik türetme
    ('scal', scaler),     # Adım 2: Normalizasyon
    ('lin',  lin_reg2)    # Adım 3: Linear Regression
])

# Pipeline'ı tam veriyle eğit
pipeline_reg.fit(Xfull, yfull)

# 0-115,000 aralığında tahmin yap
X = np.linspace(0, 115_000, 1000)
curve = pipeline_reg.predict(X[:, np.newaxis])  # np.newaxis: 1D → 2D

# Polinom eğrisini çiz
plt.plot(X, curve)

plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()
# Sonuç: Polinom eğrisi veri noktalarından geçiyor ama anlamsız dalgalanmalar var
# → Overfitting! Eğitim verisi iyi, yeni veri için kötü

## 3.16 "W" Harfi İçeren Ülkeler — Veri Kalitesi Kontrolü

Bu iki hücre, veri temizliği / veri kalitesi kontrolü için örnek bir araştırmadır.  
"W" harfi içeren ülke adlarını bularak (Taiwan vb.) farklı kaynaklarda nasıl temsil edildiklerini kontrol ediyoruz.

Bu tür kontroller gerçek dünya projelerde sık yapılır — veri kaynaklarının tutarsızlıkları (farklı ülke adı yazımları) merge sırasında kayıplara yol açabilir.


In [ ]:
# OECD BLI verisinde "W" harfi geçen ülkeleri bul
# c.upper(): Büyük/küçük harf farkını gider
# "W" in c.upper(): "W" veya "w" içeren ülkeler
w_countries = [c for c in full_country_stats.index if "W" in c.upper()]

# Bu ülkelerin yaşam memnuniyeti değerlerini göster
full_country_stats.loc[w_countries][lifesat_col]
# Örnek: "New Zealand", "Switzerland", "Norway" vb.

In [ ]:
# GDP verisinde "W" harfi geçen ülkeleri bul
all_w_countries = [c for c in gdp_per_capita.index if "W" in c.upper()]

# GDP değerlerine göre sırala (küçükten büyüğe)
gdp_per_capita.loc[all_w_countries].sort_values(by=gdppc_col)
# Karşılaştırma: GDP verisindeki ve BLI verisindeki ülke adları tutarlı mı?
# Tutarsızlık varsa merge sırasında bu ülkeler kaybolur!

## 3.17 Regularization (Düzenlileştirme)

**Regularization**, modelin karmaşıklığını kısıtlayarak overfitting'i önleme tekniğidir.

## Ridge Regression (L2 Regularization)

Standard Linear Regression'a bir **penalty terimi** ekler:

```
Cost = MSE(θ) + α × Σ θᵢ²
```

- **α (alpha)**: Regularization şiddeti
  - α=0: Normal Linear Regression (regularization yok)
  - α→∞: Tüm θ parametreleri 0'a yaklaşır (çok sert kısıt)
  - α=10^9.5: Bu örnekte çok güçlü regularization (slope neredeyse 0)

## Bu Grafik Ne Gösteriyor?

Kısmi veriden eğitilen Ridge modeli, regularization sayesinde neredeyse düz bir çizgi üretiyor.  
Bu, modelin aşırı uyum yapmasını engeller ama biraz **underfitting** riski yaratır.

**Üç modeli karşılaştırıyoruz:**
1. 🔵 Kesikli mavi: Kısmi veriden Linear Regression (yanlı tahmin)
2. ⚫ Katı siyah: Tam veriden Linear Regression (daha doğru ama veri gerekirdi)
3. 🔵-- Mavi kesme: Kısmi veriden Ridge (regularization ile overfitting önleme)


In [ ]:
# Üç modeli tek grafikte karşılaştır
# Mavi daireler: Eğitimde kullanılan ülkeler (kısmi veri)
country_stats.plot(kind='scatter', x=gdppc_col, y=lifesat_col, figsize=(8, 3))

# Kırmızı kareler: Eğitimde kullanılmayan ülkeler (eksik veri)
missing_data.plot(kind='scatter', x=gdppc_col, y=lifesat_col,
                  marker="s", color="r", grid=True, ax=plt.gca())

X = np.linspace(0, 115_000, 1000)

# ─── MODEL 1: Kısmi veriden Linear Regression ─────────────────────────────────
# Mavi noktalı çizgi — kısmi veriye iyi uyar ama eksik veriyle tutarsız
plt.plot(X, t0 + t1 * X, "b:", label="Linear model on partial data")

# ─── MODEL 2: Tam veriden Linear Regression ───────────────────────────────────
# Siyah katı çizgi — tüm veriyi görmüş, daha gerçekçi
plt.plot(X, t0full + t1full * X, "k-", label="Linear model on all data")

# ─── MODEL 3: Kısmi veriden Ridge Regression (regularized) ───────────────────
# Ridge (L2 regularization): θ parametrelerinin karelerini minimize et
# alpha=10^9.5: Çok güçlü regularization — slope neredeyse sıfıra zorlanır
# Sonuç: Neredeyse düz bir çizgi (underfitting riski var ama overfitting yok)
ridge = linear_model.Ridge(alpha=10**9.5)

# Kısmi veriyle eğit
X_sample = country_stats[[gdppc_col]]
y_sample  = country_stats[[lifesat_col]]
ridge.fit(X_sample, y_sample)

t0ridge, t1ridge = ridge.intercept_[0], ridge.coef_[0]

# Mavi kesme çizgi — regularized model
plt.plot(X, t0ridge + t1ridge * X, "b--",
         label="Regularized linear model on partial data")

plt.legend(loc="lower right")
plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()
# Yorum:
# - Mavi noktalı: Kısmi veriyi iyi öğrenmiş ama eksik ülkeler için çok iyimser
# - Siyah katı: Tüm ülkeleri gördüğü için daha gerçekçi (ama bunu hep bilemeyiz)
# - Mavi kesme: Regularization sayesinde kısmi veriyle bile daha konservatif
#   → Eksik ülkeler geldiğinde siyah çizgiye daha yakın kalıyor!

# 4. Alıştırma Çözümleri (Exercise Solutions)

Aşağıda bölüm 1'in alıştırma sorularına yanıtlar yer almaktadır.


## Soru 1: Machine Learning Nedir?

**Machine Learning**, veriden öğrenen sistemler inşa etme alanıdır.  
"Öğrenme": Belirli bir performans ölçütüne göre bir görevde daha iyi hale gelme.

---

## Soru 2: ML Nerede Parlıyor?

ML şunlar için harikadır:
- Algoritmik çözümü olmayan karmaşık problemler
- Uzun elle-ayarlı kural listelerini değiştirme
- Değişken ortamlara uyum sağlayan sistemler
- İnsanların öğrenmesine yardım etme (data mining)

---

## Soru 3: Labeled Training Set Nedir?

**Labeled training set (Etiketli eğitim seti)**: Her örnek için istenen çözümün (etikelin) bulunduğu eğitim seti.  
Örnek: Bir e-posta veri seti + her e-postanın "spam" veya "spam değil" etiketi.

---

## Soru 4: En Yaygın 2 Supervised Görev Nedir?

1. **Regression (Regresyon)**: Sayısal değer tahmini — ev fiyatı, sıcaklık
2. **Classification (Sınıflandırma)**: Kategori tahmini — spam/ham, hastalık teşhisi

---

## Soru 5: Yaygın Unsupervised Görevler Nelerdir?

- **Clustering** (Kümeleme): Benzer örnekleri gruplama
- **Visualization** (Görselleştirme): Boyut indirgeme + görsel
- **Dimensionality Reduction** (Boyut İndirgeme): Az bilgi kaybıyla özellikleri birleştirme
- **Association Rule Learning** (İlişkilendirme Kuralı): Alışveriş verisi gibi kalıplar bulma

---

## Soru 6: Robot Yürüme — Hangi ML Türü?

**Reinforcement Learning (Pekiştirmeli Öğrenme)** — en uygun seçim.  
Robot ajanı, çeşitli bilinmeyen arazilerde yürümeyi deneyip ödül/ceza alarak öğrenir.  
Supervised veya semi-supervised ile de çözülebilir ama daha doğal değil.

---

## Soru 7: Müşteri Segmentasyonu — Nasıl?

- **Grupları bilmiyorsak**: **Clustering** algoritması (unsupervised) — benzer müşterileri kümele
- **Grupları biliyorsak**: **Classification** algoritması (supervised) — her müşteriye grup ata

---

## Soru 8: Spam Tespiti — Hangi ML Türü?

**Supervised Learning** — tipik bir sınıflandırma problemi.  
Algoritma, spam ve spam olmayan e-postalarla etiketlenmiş örnekler üzerinde eğitilir.

---

## Soru 9: Online Learning Sistemi Nedir?

**Online learning**: Batch learning'in aksine modeli artımlı (incremental) olarak günceller.  
- Hızla değişen veriye uyum sağlayabilir
- Çok büyük miktarda veriyle eğitilebilir (veriyi biriktiremezsiniz)

---

## Soru 10: Out-of-Core Algoritma Nedir?

**Out-of-core (Çekirdek dışı)** algoritma: Bilgisayarın ana belleğine sığmayan çok büyük veri miktarlarını işleyebilir.  
Veriyi mini-batch'lere böler ve online learning teknikleriyle öğrenir.

---

## Soru 11: Instance-Based Learning Nedir?

**Instance-based learning sistemi**: Eğitim verisini ezberler; yeni bir örnek geldiğinde benzerlik ölçütüyle en benzer öğrenilmiş örnekleri bulur ve bunlarla tahmin yapar.  
Örnek: k-NN algoritması.

---

## Soru 12: Model Parametresi vs Hyperparameter Farkı?

**Model parametresi**: Modelin yeni bir örnek için ne tahmin edeceğini belirler  
(örn. doğrusal modelin slope değeri θ₁).  
Öğrenme algoritması bu değerleri optimize eder.

**Hyperparameter**: Öğrenme algoritmasının parametresi, modelin değil  
(örn. regularization miktarı, k-NN'deki k).  
Algoritma çalıştırılmadan önce belirlenir, eğitim sırasında değişmez.

---

## Soru 13: Model-Based Learning Nasıl Çalışır?

Model-based learning algoritmaları:
1. Model parametrelerinin optimal değerini arar
2. Eğitim verisinde iyi genelleme sağlayacak parametreler hedeflenir
3. **Cost function** minimize edilir: Eğitim hatası + karmaşıklık penaltısı (regularization varsa)
4. Tahmin için: Yeni örneğin özellikleri modelin tahmin fonksiyonuna verilir

---

## Soru 14: ML'deki Temel Zorluklar Nelerdir?

- Yetersiz veri miktarı
- Düşük veri kalitesi (gürültü, hatalar)
- Temsil edilmez veri (sampling bias)
- Bilgi vermeyen özellikler (feature engineering sorunu)
- Aşırı basit modeller → **underfitting**
- Aşırı karmaşık modeller → **overfitting**

---

## Soru 15: Eğitimde İyi, Yeni Veride Kötü Performans Neden?

Model büyük ihtimalle eğitim verisine **overfitting** yapmıştır.

**Çözümler:**
- Daha fazla veri topla
- Modeli basitleştir (daha az parametre, daha az özellik)
- Modeli regularize et
- Eğitim verisindeki gürültüyü azalt

---

## Soru 16: Test Seti Ne İçin Kullanılır?

**Test seti**: Model üretime alınmadan önce gerçek performansını (genelleme hatasını) tahmin etmek için kullanılır.

---

## Soru 17: Validation Seti Ne İçin Kullanılır?

**Validation seti**: Modelleri karşılaştırmak için kullanılır.  
En iyi modeli seçmek ve hyperparameter'ları ayarlamak (tuning) mümkün olur.

---

## Soru 18: Train-Dev Seti Nedir?

**Train-dev seti**: Eğitim verisi ile validation/test verisi arasında uyumsuzluk (data mismatch) riski olduğunda kullanılır.

- Eğitim setinin bir kısmı ayrılır (model bunu görmez)
- Model, kalan eğitim verisiyle eğitilir
- Hem train-dev hem validation'da değerlendirilir
- Train-dev'de kötü → model eğitim setine overfitting yapmış
- Train-dev'de iyi ama validation'da kötü → eğitim ve test verisi arasında data mismatch var

---

## Soru 19: Test Seti ile Hyperparameter Ayarlamanın Riski Nedir?

Test seti üzerinde hyperparameter ayarlanırsa, test setine **overfitting** riski oluşur.  
Ölçülen genelleme hatası iyimser (düşük) görünür; üretime alınan model beklenenden kötü performans gösterir.


In [ ]:
# Bu notebook'un tüm bölümleri tamamlandı.
# Bir sonraki bölüm için: 02_end_to_end_machine_learning_project.ipynb